# first model CNN-LSTM

In [6]:
"""
CNN-LSTM Best — Complete Pipeline
===================================
Built on LightCropNet Fixed, with 4 major upgrades:

1. STATE EMBEDDING  — learnable per-state vector, added to temporal feature
2. SEASON MASK V2  — Gaussian attention over growing-season weeks (per-state priors)
3. COUNTY HIST YIELD — mean historical county yield added to prediction head
4. HRRR KEY INPUT  — weather features used directly (no Sentinel required to work well)

Architecture:
  ImageEncoder (ResNet-style CNN)  →  WeatherFusion  →  SpatialPool
  → SeasonMask × TemporalGRU  →  StateEmbedding  →  PredictionHead
                                                      ↑ county_hist_yield

Paths: identical to MMST-ViT workspace structure
"""

import os, json, math, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import h5py, cv2

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    ROOT_DIR    = '/workspace/CropNet'
    TRAIN_JSON  = '/workspace/MMST-ViT/data/soybean_train.json'
    TEST_JSON   = '/workspace/MMST-ViT/data/soybean_test.json'
    OUTPUT_DIR  = '/workspace/MMST-ViT/output_dir/cnn_best'

    EPOCHS       = 50
    BATCH_SIZE   = 12
    LR           = 3e-4
    WEIGHT_DECAY = 0.05
    WARMUP       = 20
    MIN_LR       = 1e-6
    PATIENCE     = 30

    EMBED_DIM     = 256
    NUM_GRIDS     = 48
    IMG_SIZE      = 224
    NUM_SHORT     = 6
    NUM_LONG_TERM = 12
    NUM_YEAR      = 3

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    SEED   = 42

# ── State catalogue (5 soybean states in CropNet) ──────────────────────────
STATES = ['IA', 'IL', 'MN', 'NC', 'ND']
STATE_TO_IDX = {s: i for i, s in enumerate(STATES)}

# ── Season priors: (μ_week, σ_week) — week-of-year (0-indexed, 52 weeks) ──
# Based on USDA planting / reproductive calendars
SEASON_PRIORS = {
    'IA': (27.0, 6.0),   # early July
    'IL': (26.0, 6.0),   # late June
    'MN': (24.0, 5.5),   # mid June
    'ND': (23.0, 5.5),   # early June
    'NC': (30.0, 6.5),   # late July
}

# ── Weather columns ─────────────────────────────────────────────────────────
WEATHER_COLS = [
    'Avg Temperature (K)', 'Max Temperature (K)', 'Min Temperature (K)',
    'Precipitation (kg m**-2)', 'Relative Humidity (%)',
    'Wind Gust (m s**-1)', 'Wind Speed (m s**-1)',
    'U Component of Wind (m s**-1)', 'V Component of Wind (m s**-1)']
W_MEAN = np.array([290.,295.,285.,5.,50.,10.,5.,0.,0.], np.float32)
W_STD  = np.array([15., 15., 15.,10.,30., 5.,3.,5.,5.], np.float32)

# ============================================================================
# MODEL BLOCKS
# ============================================================================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_ch))
        self.skip = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                                   nn.BatchNorm2d(out_ch))
                     if stride != 1 or in_ch != out_ch else nn.Identity())
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.act(self.conv(x) + self.skip(x))


class ImageEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.stem  = nn.Sequential(
            nn.Conv2d(3, 32, 7, 2, 3, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1))
        self.l1 = ConvBlock(32,  64)
        self.l2 = ConvBlock(64,  128, stride=2)
        self.l3 = ConvBlock(128, 256, stride=2)
        self.l4 = ConvBlock(256, embed_dim, stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, x):
        x = self.stem(x)
        x = self.l1(x); x = self.l2(x); x = self.l3(x); x = self.l4(x)
        return self.norm(self.pool(x).flatten(1))


class WeatherFusion(nn.Module):
    def __init__(self, D, weather_dim=9):
        super().__init__()
        self.enc  = nn.Sequential(nn.Linear(weather_dim, D), nn.ReLU(), nn.Linear(D, D))
        self.gate = nn.Sequential(nn.Linear(D*2, D), nn.Sigmoid())
        self.proj = nn.Sequential(nn.Linear(D*2, D), nn.LayerNorm(D), nn.ReLU())
    def forward(self, vis, weather):
        w = self.enc(weather)
        c = torch.cat([vis, w], dim=-1)
        return vis + self.gate(c) * self.proj(c)


class SpatialPool(nn.Module):
    def __init__(self, D):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(D, 64), nn.Tanh(), nn.Linear(64, 1))
    def forward(self, x):          # (B, G, D)
        w = torch.softmax(self.attn(x), dim=1)
        return (w * x).sum(dim=1)  # (B, D)


class SeasonMaskV2(nn.Module):
    """
    Learnable Gaussian season mask over T weekly timesteps.
    Initialized with agronomic priors per state, then fine-tuned.
    mu and log_sigma are per-state parameters.
    """
    def __init__(self, n_states, T, priors):
        super().__init__()
        mu_init    = torch.zeros(n_states)
        lsig_init  = torch.zeros(n_states)
        for i, s in enumerate(STATES):
            mu, sig = priors[s]
            mu_init[i]   = mu / T          # normalise to [0,1]
            lsig_init[i] = math.log(sig / T)
        self.mu     = nn.Parameter(mu_init)
        self.log_sig= nn.Parameter(lsig_init)
        self.T      = T

    def forward(self, state_idx):              # state_idx: (B,)
        B = state_idx.shape[0]
        t = torch.linspace(0, 1, self.T, device=state_idx.device)  # (T,)
        mu  = self.mu[state_idx].unsqueeze(1)          # (B,1)
        sig = torch.exp(self.log_sig[state_idx]).unsqueeze(1) + 1e-6  # (B,1)
        mask = torch.exp(-0.5 * ((t - mu) / sig) ** 2)  # (B,T)
        mask = mask / (mask.sum(dim=1, keepdim=True) + 1e-6)
        return mask   # (B,T) — attention weights over timesteps


class TemporalGRU(nn.Module):
    def __init__(self, D, n_layers=2, dropout=0.1):
        super().__init__()
        self.gru  = nn.GRU(D, D, n_layers, batch_first=True,
                           dropout=dropout if n_layers > 1 else 0)
        self.norm = nn.LayerNorm(D)
    def forward(self, x, bias=None):
        if bias is not None:
            x = x + bias.unsqueeze(1)
        out, _ = self.gru(x)
        return self.norm(out)   # (B, T, D)


class PredictionHead(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc1  = nn.Linear(in_dim, 256)
        self.fc2  = nn.Linear(256, 128)
        self.fc3  = nn.Linear(128, 1)
        self.norm = nn.LayerNorm(256)
        self.drop = nn.Dropout(0.1)
    def forward(self, x):
        h = F.relu(self.norm(self.fc1(x)))
        h = self.drop(h)
        h = F.relu(self.fc2(h))
        return self.fc3(h)


# ── Main model ───────────────────────────────────────────────────────────────
class CNNBest(nn.Module):
    """
    CNN-LSTM with:
      • State embedding (learnable, per state)
      • SeasonMask V2 (Gaussian, per-state agronomic priors)
      • County historical yield as extra head input
    """
    def __init__(self, config):
        super().__init__()
        D  = config.EMBED_DIM
        NS = len(STATES)
        T  = config.NUM_SHORT

        self.image_enc   = ImageEncoder(D)
        self.weather_fus = WeatherFusion(D, 9)
        self.spatial_pool= SpatialPool(D)

        long_in = config.NUM_YEAR * config.NUM_LONG_TERM * 9
        self.long_enc = nn.Sequential(
            nn.Linear(long_in, D*2), nn.ReLU(),
            nn.Dropout(0.1), nn.Linear(D*2, D), nn.LayerNorm(D))

        self.season_mask   = SeasonMaskV2(NS, T, SEASON_PRIORS)
        self.temporal_gru  = TemporalGRU(D, n_layers=2, dropout=0.1)
        self.state_embed   = nn.Embedding(NS, D)

        # head input: temporal_agg (D) + long_bias (D) + state_emb (D) + county_hist (1)
        self.head = PredictionHead(D * 3 + 1)

    def forward(self, sentinel, hrrr_short, hrrr_long, state_idx, county_hist):
        B, T, G, C, H, W = sentinel.shape

        # ── image+weather per (t, g) ────────────────────────────────
        imgs    = sentinel.view(B*T*G, C, H, W)
        visual  = self.image_enc(imgs).view(B, T, G, -1)
        weather = hrrr_short.mean(dim=3)               # (B,T,G,9)
        fused   = self.weather_fus(
            visual.view(B*T*G, -1),
            weather.view(B*T*G, 9)).view(B, T, G, -1)

        # ── spatial pooling ─────────────────────────────────────────
        spatial = self.spatial_pool(fused.view(B*T, G, -1)).view(B, T, -1)  # (B,T,D)

        # ── long-term climate bias ──────────────────────────────────
        long_bias = self.long_enc(hrrr_long.view(B, -1))   # (B,D)

        # ── temporal GRU ────────────────────────────────────────────
        gru_out = self.temporal_gru(spatial, bias=long_bias)  # (B,T,D)

        # ── season mask (per-state Gaussian attention) ──────────────
        mask = self.season_mask(state_idx)      # (B,T)
        temporal_agg = (mask.unsqueeze(-1) * gru_out).sum(dim=1)  # (B,D)

        # ── state embedding ─────────────────────────────────────────
        st_emb = self.state_embed(state_idx)    # (B,D)

        # ── prediction head ─────────────────────────────────────────
        ch = county_hist.unsqueeze(-1)          # (B,1)
        feat = torch.cat([temporal_agg, long_bias, st_emb, ch], dim=-1)
        return self.head(feat).squeeze(-1)      # (B,)


# ============================================================================
# DATASET
# ============================================================================
class YieldDataset(Dataset):
    def __init__(self, root_dir, json_file, config,
                 yield_mean=None, yield_std=None,
                 county_hist=None):
        self.root_dir   = root_dir
        self.cfg        = config
        self.ym         = yield_mean
        self.ys         = yield_std
        self.county_hist= county_hist or {}   # {fips: mean_yield}
        self.samples    = self._load(json_file)

    def _load(self, json_file):
        print(f'Loading {json_file}...')
        with open(json_file) as f:
            data = json.load(f)
        valid = []
        for s in tqdm(data, desc='Validating', leave=False):
            try:
                usda = os.path.join(self.root_dir, s['data']['USDA'])
                if not os.path.exists(usda): continue
                df = pd.read_csv(usda)
                if 'FIPS' not in df.columns:
                    df['FIPS'] = (df['state_ansi'].astype(str).str.zfill(2) +
                                  df['county_ansi'].astype(str).str.zfill(3))
                row = df[df['FIPS'] == str(s['FIPS'])]
                if len(row) == 0: continue
                y = float(row['YIELD, MEASURED IN BU / ACRE'].values[0])
                if pd.isna(y) or y <= 0: continue
                s['yield'] = y
                # extract state abbreviation from FIPS
                st = s.get('state', '') or s.get('State', '')
                if not st:
                    st = self._fips_to_state(str(s['FIPS'])[:2])
                s['state_abbr'] = st.upper()[:2]
                valid.append(s)
            except Exception:
                continue
        print(f'  ✅ {len(valid)} valid samples')
        return valid

    @staticmethod
    def _fips_to_state(fips2):
        m = {'19':'IA','17':'IL','27':'MN','37':'NC','38':'ND'}
        return m.get(fips2, 'IA')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s          = self.samples[idx]
        sentinel   = self._sentinel(s)
        hrrr_short = self._hrrr_short(s)
        hrrr_long  = self._hrrr_long(s)
        y_raw      = s['yield']
        y_norm     = (y_raw - self.ym) / self.ys if self.ym else y_raw

        st  = s['state_abbr']
        sid = STATE_TO_IDX.get(st, 0)

        fips = str(s['FIPS'])
        ch   = self.county_hist.get(fips, self.ym or 50.0)
        ch_norm = (ch - (self.ym or 0)) / (self.ys or 1)

        return (sentinel, hrrr_short, hrrr_long,
                torch.tensor(y_norm,  dtype=torch.float32),
                torch.tensor(y_raw,   dtype=torch.float32),
                torch.tensor(sid,     dtype=torch.long),
                torch.tensor(ch_norm, dtype=torch.float32))

    # ── image loading (same robust logic as LightCropNet fixed) ─────────
    def _sentinel(self, s):
        paths = s['data'].get('Sentinel-2', {})
        if isinstance(paths, dict): paths = list(paths.values())
        if isinstance(paths, str):  paths = [paths]
        grids = []
        for p in paths[:self.cfg.NUM_GRIDS]:
            try:   grids.append(self._load_grid(self._fp(p)))
            except: grids.append(np.zeros((3,self.cfg.IMG_SIZE,self.cfg.IMG_SIZE),np.float32))
        while len(grids) < self.cfg.NUM_GRIDS:
            grids.append(np.zeros((3,self.cfg.IMG_SIZE,self.cfg.IMG_SIZE),np.float32))
        t = torch.from_numpy(np.stack(grids[:self.cfg.NUM_GRIDS])).float() / 10000.
        t = torch.clamp(t, 0, 1)
        return t.unsqueeze(0).repeat(self.cfg.NUM_SHORT, 1, 1, 1, 1)

    def _load_grid(self, path):
        if not os.path.exists(path):
            return np.zeros((3,self.cfg.IMG_SIZE,self.cfg.IMG_SIZE),np.float32)
        with h5py.File(path,'r') as f:
            k = list(f.keys())
            if not k: raise ValueError
            d = f[k[0]][:]
        while d.ndim > 3: d = d[0]
        if d.ndim == 2: d = d[np.newaxis]
        if d.ndim == 3 and d.shape[2] in (3,4): d = np.transpose(d,(2,0,1))
        d = d.astype(np.float32)
        if d.shape[0] > 3: d = d[:3]
        elif d.shape[0] < 3:
            d = np.concatenate([d, np.zeros((3-d.shape[0],d.shape[1],d.shape[2]),np.float32)],0)
        H,W = self.cfg.IMG_SIZE, self.cfg.IMG_SIZE
        if d.shape[1]!=H or d.shape[2]!=W:
            d = np.stack([cv2.resize(d[c],(W,H)) for c in range(3)])
        return d

    def _hrrr_short(self, s):
        paths = s['data'].get('WRF-HRRR Computed Dataset',{}).get('short_term',[])
        if isinstance(paths,str): paths=[paths]
        grids=[]
        for p in paths[:self.cfg.NUM_GRIDS]:
            try:
                df = pd.read_csv(self._fp(p))
                avail = [c for c in WEATHER_COLS if c in df.columns]
                v = df[avail].head(28).fillna(0).values if avail else np.zeros((28,9),np.float32)
                if v.shape[1]<9: v=np.pad(v,((0,0),(0,9-v.shape[1])))
                v = (v[:28] if len(v)>=28 else np.pad(v,((0,28-len(v)),(0,0)))).astype(np.float32)
                grids.append(v)
            except: grids.append(np.zeros((28,9),np.float32))
        while len(grids)<self.cfg.NUM_GRIDS: grids.append(np.zeros((28,9),np.float32))
        t = torch.from_numpy(np.stack(grids[:self.cfg.NUM_GRIDS])).float()
        t = (t - torch.tensor(W_MEAN).view(1,1,-1)) / torch.tensor(W_STD).view(1,1,-1)
        return t.unsqueeze(0).repeat(self.cfg.NUM_SHORT,1,1,1)

    def _hrrr_long(self, s):
        ld = s['data'].get('WRF-HRRR Computed Dataset',{}).get('long_term',[])
        res=[]
        for yr in ld[:self.cfg.NUM_YEAR]:
            yr_v=[]
            if isinstance(yr,list):
                for mp in yr[:self.cfg.NUM_LONG_TERM]:
                    try:
                        df=pd.read_csv(self._fp(mp))
                        avail=[c for c in WEATHER_COLS if c in df.columns]
                        v=df[avail].mean(0).values if avail else np.zeros(9)
                        if len(v)<9: v=np.pad(v,(0,9-len(v)))
                        yr_v.append(v[:9].astype(np.float32))
                    except: yr_v.append(np.zeros(9,np.float32))
            while len(yr_v)<self.cfg.NUM_LONG_TERM: yr_v.append(np.zeros(9,np.float32))
            res.append(yr_v[:self.cfg.NUM_LONG_TERM])
        while len(res)<self.cfg.NUM_YEAR:
            res.append([np.zeros(9,np.float32)]*self.cfg.NUM_LONG_TERM)
        t = torch.from_numpy(np.array(res)).float()
        t = (t - torch.tensor(W_MEAN).view(1,1,-1)) / torch.tensor(W_STD).view(1,1,-1)
        return t

    def _fp(self, p):
        full = os.path.join(self.root_dir, p)
        if os.path.exists(full): return full
        for i in range(len(p.split('/'))):
            c = os.path.join(self.root_dir, *p.split('/')[i:])
            if os.path.exists(c): return c
        return full


# ============================================================================
# COUNTY HISTORICAL YIELD HELPER
# ============================================================================
def build_county_hist(samples):
    """Average yield per FIPS across ALL years in training set."""
    from collections import defaultdict
    acc = defaultdict(list)
    for s in samples:
        acc[str(s['FIPS'])].append(s['yield'])
    return {fips: float(np.mean(vals)) for fips, vals in acc.items()}


# ============================================================================
# TRAINING UTILITIES
# ============================================================================
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def cosine_lr(optimizer, epoch, cfg):
    if epoch < cfg.WARMUP:
        lr = cfg.LR * (epoch + 1) / cfg.WARMUP
    else:
        p  = (epoch - cfg.WARMUP) / max(cfg.EPOCHS - cfg.WARMUP, 1)
        lr = cfg.MIN_LR + 0.5*(cfg.LR - cfg.MIN_LR)*(1 + math.cos(math.pi*p))
    for pg in optimizer.param_groups: pg['lr'] = lr
    return lr


def train_epoch(model, loader, criterion, optimizer, device):
    model.train(); total = 0.
    for batch in tqdm(loader, desc='Train', leave=False):
        sentinel, hrrr_s, hrrr_l, y_norm, _, sid, ch = batch
        sentinel=sentinel.to(device); hrrr_s=hrrr_s.to(device)
        hrrr_l=hrrr_l.to(device); y_norm=y_norm.to(device)
        sid=sid.to(device); ch=ch.to(device)

        optimizer.zero_grad()
        out  = model(sentinel, hrrr_s, hrrr_l, sid, ch)
        loss = criterion(out, y_norm)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader, device, ym, ys):
    model.eval()
    preds, trues, states = [], [], []
    for batch in tqdm(loader, desc='Eval', leave=False):
        sentinel, hrrr_s, hrrr_l, _, y_raw, sid, ch = batch
        sentinel=sentinel.to(device); hrrr_s=hrrr_s.to(device)
        hrrr_l=hrrr_l.to(device); sid_d=sid.to(device); ch_d=ch.to(device)

        out = model(sentinel, hrrr_s, hrrr_l, sid_d, ch_d).cpu().numpy()
        preds.extend(out * ys + ym)
        trues.extend(y_raw.numpy())
        states.extend(sid.numpy())

    p = np.array(preds); t = np.array(trues); st = np.array(states)
    rmse = float(np.sqrt(((p-t)**2).mean()))
    mae  = float(np.abs(p-t).mean())
    ss_r = float(np.sum((t-p)**2))
    ss_t = float(np.sum((t-t.mean())**2))
    r2   = 1 - ss_r/ss_t if ss_t > 0 else 0.
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        corr, _ = pearsonr(p, t)
        corr = float(corr) if not np.isnan(corr) else 0.

    return rmse, mae, r2, corr, p, t, st


# ============================================================================
# PLOTS
# ============================================================================
def plot_training(history, out_dir):
    fig, axes = plt.subplots(1, 3, figsize=(16,5))
    axes[0].plot(history['loss'], color='#457b9d', lw=2)
    axes[0].set_title('Huber Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=.3)

    axes[1].plot(history['rmse'], color='#e63946', lw=2, label='RMSE')
    axes[1].plot(history['mae'],  color='#f4a261', lw=2, label='MAE')
    axes[1].set_title('RMSE & MAE (bu/acre)'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(alpha=.3)

    axes[2].plot(history['r2'],   color='#2a9d8f', lw=2, label='R²')
    axes[2].plot(history['corr'], color='#e9c46a', lw=2, label='Pearson r')
    axes[2].set_title('R² & Pearson r'); axes[2].set_xlabel('Epoch')
    axes[2].legend(); axes[2].grid(alpha=.3)

    plt.suptitle('CNN-Best — Training Curves', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir,'training_curves.png'), dpi=150); plt.close()


def plot_results(preds, targets, st_arr, rmse, r2, corr, out_dir):
    colors  = ['#e63946','#457b9d','#2a9d8f','#f4a261','#8338ec']
    fig, axes = plt.subplots(1, 2, figsize=(14,6))

    for i, s in enumerate(STATES):
        mask = st_arr == i
        if mask.sum() == 0: continue
        axes[0].scatter(targets[mask], preds[mask], alpha=.55, s=20,
                        color=colors[i], label=s)

    lim = [min(targets.min(),preds.min())-3, max(targets.max(),preds.max())+3]
    axes[0].plot(lim, lim, 'k--', lw=1.5)
    axes[0].set_xlim(lim); axes[0].set_ylim(lim)
    axes[0].set_xlabel('Actual (bu/acre)'); axes[0].set_ylabel('Predicted (bu/acre)')
    axes[0].set_title(f'Pred vs Actual\nRMSE={rmse:.2f}  R²={r2:.4f}  r={corr:.4f}')
    axes[0].legend(title='State', fontsize=9); axes[0].grid(alpha=.3)

    # per-state RMSE bar
    per_rmse = []
    for i, s in enumerate(STATES):
        mask = st_arr == i
        if mask.sum() < 2:
            per_rmse.append(0.)
        else:
            per_rmse.append(float(np.sqrt(((preds[mask]-targets[mask])**2).mean())))
    axes[1].bar(STATES, per_rmse, color=colors)
    axes[1].set_xlabel('State'); axes[1].set_ylabel('RMSE (bu/acre)')
    axes[1].set_title('Per-State RMSE'); axes[1].grid(alpha=.3, axis='y')

    plt.suptitle('CNN-Best — Final Results', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir,'results.png'), dpi=150); plt.close()


# ============================================================================
# MAIN
# ============================================================================
def main():
    set_seed(Config.SEED)
    os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
    CKPT = os.path.join(Config.OUTPUT_DIR, 'best_model.pth')

    print('='*70)
    print('  CNN-Best  —  CNN-LSTM + State Embedding + Season Mask + County Hist')
    print('='*70)
    print(f'  Device    : {Config.DEVICE}')
    print(f'  Epochs    : {Config.EPOCHS}  (early-stop patience={Config.PATIENCE})')
    print(f'  Batch     : {Config.BATCH_SIZE}')
    print('='*70)

    # ── 1. Compute target normalization & county hist from train ──────────
    print('\n[1/5] Computing stats from train set...')
    tmp = YieldDataset(Config.ROOT_DIR, Config.TRAIN_JSON, Config)
    raw_yields   = np.array([s['yield'] for s in tmp.samples])
    YIELD_MEAN   = float(raw_yields.mean())
    YIELD_STD    = float(raw_yields.std())
    county_hist  = build_county_hist(tmp.samples)
    print(f'  yield mean={YIELD_MEAN:.2f}  std={YIELD_STD:.2f}  counties={len(county_hist)}')

    # ── 2. Datasets ───────────────────────────────────────────────────────
    print('\n[2/5] Building datasets...')
    train_ds = YieldDataset(Config.ROOT_DIR, Config.TRAIN_JSON, Config,
                            YIELD_MEAN, YIELD_STD, county_hist)
    test_ds  = YieldDataset(Config.ROOT_DIR, Config.TEST_JSON,  Config,
                            YIELD_MEAN, YIELD_STD, county_hist)

    train_loader = DataLoader(train_ds, Config.BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_ds,  Config.BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)

    # ── 3. Model ──────────────────────────────────────────────────────────
    print('\n[3/5] Building model...')
    model  = CNNBest(Config).to(Config.DEVICE)
    n_par  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Parameters: {n_par/1e6:.2f}M')

    criterion = nn.HuberLoss(delta=1.0)
    optimizer = optim.AdamW(model.parameters(), lr=Config.LR,
                            betas=(0.9,0.95), weight_decay=Config.WEIGHT_DECAY)

    # ── 4. Training loop ──────────────────────────────────────────────────
    print('\n[4/5] Training...\n')
    best_rmse, patience_cnt = float('inf'), 0
    history = {'loss':[],'rmse':[],'mae':[],'r2':[],'corr':[],'lr':[]}

    for epoch in range(Config.EPOCHS):
        lr         = cosine_lr(optimizer, epoch, Config)
        train_loss = train_epoch(model, train_loader, criterion, optimizer, Config.DEVICE)
        rmse,mae,r2,corr,_,_,_ = evaluate(model, test_loader, Config.DEVICE,
                                           YIELD_MEAN, YIELD_STD)

        history['loss'].append(train_loss); history['rmse'].append(rmse)
        history['mae'].append(mae);         history['r2'].append(r2)
        history['corr'].append(corr);       history['lr'].append(lr)

        flag = ''
        if rmse < best_rmse:
            best_rmse, patience_cnt = rmse, 0
            torch.save({'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'rmse': rmse, 'r2': r2,
                        'yield_mean': YIELD_MEAN, 'yield_std': YIELD_STD,
                        'county_hist': county_hist}, CKPT)
            flag = ' ✅ best'
        else:
            patience_cnt += 1

        print(f'Ep {epoch:>3} | Loss {train_loss:.4f} | '
              f'RMSE {rmse:.2f} | MAE {mae:.2f} | '
              f'R² {r2:.4f} | r {corr:.4f} | LR {lr:.2e}{flag}')

        if patience_cnt >= Config.PATIENCE:
            print(f'\n⏹ Early stop at epoch {epoch}')
            break

    # ── 5. Final evaluation ───────────────────────────────────────────────
    print('\n[5/5] Final evaluation...')
    ckpt = torch.load(CKPT, map_location=Config.DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    rmse,mae,r2,corr,preds,targets,st_arr = evaluate(
        model, test_loader, Config.DEVICE, YIELD_MEAN, YIELD_STD)

    plot_training(history, Config.OUTPUT_DIR)
    plot_results(preds, targets, st_arr, rmse, r2, corr, Config.OUTPUT_DIR)

    print(f'\n{"="*70}')
    print('   FINAL RESULTS — CNN-Best')
    print(f'{"="*70}')
    print(f'  RMSE      : {rmse:.4f} bu/acre')
    print(f'  MAE       : {mae:.4f} bu/acre')
    print(f'  R²        : {r2:.4f}')
    print(f'  Pearson r : {corr:.4f}')
    print()
    print(f'  {"State":>6} | {"RMSE":>6} | {"MAE":>6} | {"R²":>8} | {"n":>5}')
    print('  ' + '-'*42)
    for i, s in enumerate(STATES):
        mask = st_arr == i
        if mask.sum() < 2: continue
        p_s = preds[mask]; t_s = targets[mask]
        rmse_s = float(np.sqrt(((p_s-t_s)**2).mean()))
        mae_s  = float(np.abs(p_s-t_s).mean())
        ss_r_s = float(np.sum((t_s-p_s)**2))
        ss_t_s = float(np.sum((t_s-t_s.mean())**2))
        r2_s   = 1 - ss_r_s/ss_t_s if ss_t_s>0 else 0.
        print(f'  {s:>6} | {rmse_s:>6.2f} | {mae_s:>6.2f} | {r2_s:>8.4f} | {mask.sum():>5}')

    print()
    print('  Learned season centers (SeasonMaskV2):')
    print(f'  {"State":>6} | {"μ (week)":>9} | {"σ (weeks)":>10} | {"Peak month":>12}')
    print('  ' + '-'*46)
    import calendar
    with torch.no_grad():
        for i, s in enumerate(STATES):
            mu  = float(model.season_mask.mu[i].item())
            sig = float(torch.exp(model.season_mask.log_sig[i]).item())
            week = mu * Config.NUM_SHORT       # back to week scale
            day  = int(week * 7)
            month = min(12, max(1, int(day/30)+1))
            print(f'  {s:>6} | {week*7/7:>9.1f} | {sig*Config.NUM_SHORT:>10.1f} | '
                  f'{calendar.month_abbr[month]:>12}')

    print(f'\n  Parameters  : {n_par/1e6:.2f}M  (MMST-ViT was 43.2M)')
    print(f'  Best epoch  : {ckpt["epoch"]}')
    print(f'  Outputs     : {Config.OUTPUT_DIR}')
    print(f'{"="*70}')


if __name__ == '__main__':
    main()


  CNN-Best  —  CNN-LSTM + State Embedding + Season Mask + County Hist
  Device    : cuda
  Epochs    : 50  (early-stop patience=30)
  Batch     : 12

[1/5] Computing stats from train set...
Loading /workspace/MMST-ViT/data/soybean_train.json...


  ✅ 344 valid samples
  yield mean=48.52  std=15.85  counties=344

[2/5] Building datasets...
Loading /workspace/MMST-ViT/data/soybean_train.json...


  ✅ 344 valid samples
Loading /workspace/MMST-ViT/data/soybean_test.json...


  ✅ 327 valid samples

[3/5] Building model...
  Parameters: 4.13M

[4/5] Training...



Ep   0 | Loss 0.3598 | RMSE 9.61 | MAE 8.03 | R² 0.4542 | r 0.7753 | LR 1.50e-05 ✅ best


...

Ep  46 | Loss 0.0033 | RMSE 6.96 | MAE 5.21 | R² 0.7139 | r 0.8982 | LR 1.39e-05


Ep  47 | Loss 0.0036 | RMSE 6.92 | MAE 5.17 | R² 0.7168 | r 0.8987 | LR 8.32e-06


Ep  48 | Loss 0.0031 | RMSE 6.99 | MAE 5.23 | R² 0.7110 | r 0.8992 | LR 4.27e-06


Ep  49 | Loss 0.0031 | RMSE 6.89 | MAE 5.16 | R² 0.7196 | r 0.8996 | LR 1.82e-06

[5/5] Final evaluation...



   FINAL RESULTS — CNN-Best
  RMSE      : 6.2949 bu/acre
  MAE       : 5.0009 bu/acre
  R²        : 0.7660
  Pearson r : 0.8831

   State |   RMSE |    MAE |       R² |     n
  ------------------------------------------
      IA |   7.20 |   5.17 |   0.0339 |    75
      IL |   5.02 |   4.22 |   0.5344 |    74
      MN |   5.90 |   4.59 |   0.5867 |    66
      NC |   5.73 |   4.89 |   0.1386 |    71
      ND |   7.93 |   6.94 |  -0.1735 |    41

  Learned season centers (SeasonMaskV2):
   State |  μ (week) |  σ (weeks) |   Peak month
  ----------------------------------------------
      IA |      26.8 |        6.1 |          Jul
      IL |      25.8 |        6.1 |          Jul
      MN |      23.8 |        5.6 |          Jun
      NC |      29.8 |        6.5 |          Jul
      ND |      22.8 |        5.5 |          Jun

  Parameters  : 4.13M  (MMST-ViT was 43.2M)
  Best epoch  : 23
  Outputs     : /workspace/MMST-ViT/output_dir/cnn_best


# Second Cnn model 

In [ ]:
"""
CNN-Best V2.1  —  Fixed version of V2
======================================================================
  What was wrong in V2
  --------------------

  What is kept from V2 (all the real improvements)
  -------------------------------------------------
  ✅ Weather 1D-CNN  — full 28-day sequence per grid
  ✅ Year embedding  — learnable token for temporal trend
  ✅ County features — mean / std / trend fed as aux inputs to head
  ✅ State-weighted loss — IA/ND × 1.5, NC × 1.3
  ✅ dropout=0.2, weight_decay=0.1
  ✅ SeasonMaskV2, TemporalGRU, SpatialPool
  ✅ BATCH=12, PATIENCE=20, WARMUP=10, EPOCHS=50
  ✅ All 12 diagnostic plots
======================================================================
"""

import os, json, math, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import h5py, cv2

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    ROOT_DIR      = '/workspace/CropNet'
    TRAIN_JSON    = '/workspace/MMST-ViT/data/soybean_train.json'
    TEST_JSON     = '/workspace/MMST-ViT/data/soybean_test.json'
    OUTPUT_DIR    = '/workspace/MMST-ViT/output_dir/cnn_best_v2_1'

    EPOCHS        = 50
    BATCH_SIZE    = 12
    LEARNING_RATE = 3e-4
    WEIGHT_DECAY  = 0.1
    WARMUP_EPOCHS = 10
    MIN_LR        = 1e-6
    PATIENCE      = 20

    EMBED_DIM     = 256
    NUM_GRIDS     = 64
    IMG_SIZE      = 224
    NUM_SHORT_TERM  = 6
    NUM_LONG_TERM   = 12
    NUM_YEAR        = 3

    YEAR_BASE       = 2000
    YEAR_EMBED_DIM  = 32

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    SEED   = 42

    STATE_WEIGHTS = {'IA': 1.5, 'IL': 1.0, 'MN': 1.0, 'NC': 1.3, 'ND': 1.5}
    STATE_ORDER   = ['IA', 'IL', 'MN', 'NC', 'ND']
    STATE_COLORS  = {
        'IA': '#e63946', 'IL': '#457b9d',
        'MN': '#2a9d8f', 'NC': '#f4a261', 'ND': '#8338ec',
    }

# ============================================================================
# WEATHER CONSTANTS
# ============================================================================
WEATHER_COLS = [
    'Avg Temperature (K)', 'Max Temperature (K)', 'Min Temperature (K)',
    'Precipitation (kg m**-2)', 'Relative Humidity (%)',
    'Wind Gust (m s**-1)', 'Wind Speed (m s**-1)',
    'U Component of Wind (m s**-1)', 'V Component of Wind (m s**-1)',
]
W_MEAN = np.array([290., 295., 285., 5., 50., 10., 5., 0., 0.], dtype=np.float32)
W_STD  = np.array([15.,  15.,  15., 10., 30.,  5., 3., 5., 5.], dtype=np.float32)

STATE_MAP = {'IA': 0, 'IL': 1, 'MN': 2, 'NC': 3, 'ND': 4}

# ============================================================================
# BUILDING BLOCKS
# ============================================================================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_ch))
        self.skip = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                                   nn.BatchNorm2d(out_ch))
                     if stride != 1 or in_ch != out_ch else nn.Identity())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.conv(x) + self.skip(x))


class ImageEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.stem   = nn.Sequential(
            nn.Conv2d(3, 32, 7, 2, 3, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1))
        self.layer1 = ConvBlock(32,  64)
        self.layer2 = ConvBlock(64,  128, stride=2)
        self.layer3 = ConvBlock(128, 256, stride=2)
        self.layer4 = ConvBlock(256, embed_dim, stride=2)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.norm   = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        return self.norm(self.pool(x).flatten(1))


class Weather1DCNN(nn.Module):
    """
    Input : (N, 28, 9)  — 28 days × 9 weather vars
    Output: (N, embed_dim)
    """
    def __init__(self, embed_dim, weather_dim=9, dropout=0.2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(weather_dim, 64,  3, padding=1),
            nn.BatchNorm1d(64),        nn.ReLU(inplace=True),
            nn.Conv1d(64,  128, 3, padding=1),
            nn.BatchNorm1d(128),       nn.ReLU(inplace=True),
            nn.Conv1d(128, embed_dim, 3, padding=1),
            nn.BatchNorm1d(embed_dim), nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):          # x: (N, 28, 9)
        x = x.transpose(1, 2)     # (N, 9, 28)
        x = self.conv(x)           # (N, D, 28)
        x = self.pool(x).squeeze(-1)
        return self.norm(self.drop(x))   # (N, D)


class WeatherFusion(nn.Module):
    def __init__(self, embed_dim, dropout=0.2):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(embed_dim * 2, embed_dim), nn.Sigmoid())
        self.proj = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.LayerNorm(embed_dim), nn.ReLU(), nn.Dropout(dropout))

    def forward(self, visual, weather_feat):
        combined = torch.cat([visual, weather_feat], dim=-1)
        return visual + self.gate(combined) * self.proj(combined)


class SpatialPool(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(embed_dim, 64), nn.Tanh(), nn.Linear(64, 1))

    def forward(self, x):   # (B, G, D)
        w = torch.softmax(self.attn(x), dim=1)
        return (w * x).sum(dim=1)   # (B, D)


class TemporalGRU(nn.Module):
    def __init__(self, embed_dim, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru  = nn.GRU(embed_dim, embed_dim, num_layers,
                           batch_first=True,
                           dropout=dropout if num_layers > 1 else 0)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, bias=None):
        if bias is not None:
            x = x + bias.unsqueeze(1)
        out, _ = self.gru(x)
        return self.norm(out)


class SeasonMaskV2(nn.Module):
    """Learnable Gaussian season mask initialized from agronomic priors."""
    def __init__(self, num_states, num_weeks=52):
        super().__init__()
        # IA IL MN NC ND  — week-of-year priors
        self.mu        = nn.Parameter(torch.tensor(
            [26., 26., 24., 30., 23.], dtype=torch.float32))
        self.log_sigma = nn.Parameter(torch.zeros(num_states))
        self.num_weeks = num_weeks

    def forward(self, state_ids):   # (B,)
        weeks = torch.arange(1, self.num_weeks + 1,
                             dtype=torch.float32, device=self.mu.device)
        mu    = self.mu[state_ids].unsqueeze(1)
        sigma = self.log_sigma[state_ids].exp().unsqueeze(1) + 1e-6
        mask  = torch.exp(-0.5 * ((weeks - mu) / sigma) ** 2)
        return mask / (mask.sum(dim=1, keepdim=True) + 1e-8)   # (B, 52)


class PredictionHead(nn.Module):
    """
    Predicts normalised yield directly: y_norm = (y - YIELD_MEAN) / YIELD_STD
    Input dim = D*2 + YEAR_EMBED_DIM + 32 (county proj)
    """
    def __init__(self, in_dim, dropout=0.2):
        super().__init__()
        self.fc1  = nn.Linear(in_dim, 256)
        self.norm = nn.LayerNorm(256)
        self.drop = nn.Dropout(dropout)
        self.fc2  = nn.Linear(256, 128)
        self.fc3  = nn.Linear(128, 1)

    def forward(self, x):
        h = F.relu(self.norm(self.fc1(x)))
        h = self.drop(h)
        h = F.relu(self.fc2(h))
        return self.fc3(h)


# ============================================================================
# MAIN MODEL — CNN-Best V2.1
# ============================================================================
class CNNBestV2(nn.Module):
    """
    CNN-Best V2.1 — keeps all V2 improvements, fixes anomaly target bug.

    Forward inputs
    --------------
    sentinel     : (B, T, G, 3, H, W)
    hrrr_short   : (B, T, G, 28, 9)
    hrrr_long    : (B, Y, M, 9)
    state_ids    : (B,)  long
    year_idx     : (B,)  long   (year - YEAR_BASE)
    county_feats : (B, 3) float (norm_mean, norm_std, trend) — AUX inputs only

    Output: (B,)  normalised yield = (y_raw - YIELD_MEAN) / YIELD_STD
    """
    def __init__(self, config):
        super().__init__()
        D  = config.EMBED_DIM
        Dy = config.YEAR_EMBED_DIM
        NS = len(config.STATE_ORDER)

        self.image_encoder  = ImageEncoder(D)
        self.weather_1d_cnn = Weather1DCNN(D, dropout=0.2)
        self.weather_fusion = WeatherFusion(D, dropout=0.2)
        self.spatial_pool   = SpatialPool(D)

        long_in = config.NUM_YEAR * config.NUM_LONG_TERM * 9
        self.long_term_enc  = nn.Sequential(
            nn.Linear(long_in, D * 2), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(D * 2, D), nn.LayerNorm(D))

        self.temporal_gru = TemporalGRU(D, num_layers=2, dropout=0.2)
        self.season_mask  = SeasonMaskV2(num_states=NS)

        self.year_embed   = nn.Embedding(30, Dy)   # years 2000-2029

        # County features: (norm_mean, norm_std, trend) → 32-d projection
        self.county_proj  = nn.Sequential(
            nn.Linear(3, 32), nn.ReLU(), nn.LayerNorm(32))

        # head: temporal_agg(D) + long_bias(D) + year_emb(Dy) + county(32)
        head_in = D * 2 + Dy + 32
        self.head = PredictionHead(head_in, dropout=0.2)

    def forward(self, sentinel, hrrr_short, hrrr_long,
                state_ids, year_idx, county_feats):
        B, T, G, C, H, W = sentinel.shape

        # ── Image stream ──────────────────────────────────────────────
        imgs   = sentinel.view(B * T * G, C, H, W)
        visual = self.image_encoder(imgs).view(B, T, G, -1)  # (B, T, G, D)

        # ── Weather 1D-CNN stream ─────────────────────────────────────
        # use the first timestep's weather (grids are the same across T)
        w_seq  = hrrr_short[:, 0, :, :, :]           # (B, G, 28, 9)
        w_feat = self.weather_1d_cnn(
            w_seq.reshape(B * G, 28, 9))              # (B*G, D)
        w_feat = w_feat.view(B, G, -1)                # (B, G, D)

        # ── FIX: average visual over T before fusion ──────────────────
        # visual.mean(dim=1) → (B, G, D)  [was: .mean(dim=0) ← BUG]
        visual_mean = visual.mean(dim=1)              # (B, G, D)
        fused = self.weather_fusion(
            visual_mean.reshape(B * G, -1),
            w_feat.reshape(B * G, -1)
        ).view(B, G, -1)                              # (B, G, D)

        # ── Spatial pooling ───────────────────────────────────────────
        spatial = self.spatial_pool(fused)            # (B, D)

        # ── Long-term climate bias ─────────────────────────────────────
        long_bias = self.long_term_enc(hrrr_long.view(B, -1))   # (B, D)

        # ── Temporal GRU — expand spatial over T ──────────────────────
        temporal_in  = spatial.unsqueeze(1).expand(-1, T, -1)   # (B, T, D)
        temporal_out = self.temporal_gru(temporal_in, bias=long_bias)

        # ── Season mask (Gaussian attention over T timesteps) ──────────
        # Note: mask is over num_weeks=52; we adapt to T with simple avg
        mask         = self.season_mask(state_ids)[:, :T]        # (B, T)
        mask         = mask / (mask.sum(dim=1, keepdim=True) + 1e-8)
        temporal_agg = (mask.unsqueeze(-1) * temporal_out).sum(dim=1)  # (B, D)

        # ── Year embedding ────────────────────────────────────────────
        yr_emb = self.year_embed(year_idx)    # (B, Dy)

        # ── County auxiliary features ─────────────────────────────────
        c_emb  = self.county_proj(county_feats)   # (B, 32)

        # ── Head: predict normalised yield directly ────────────────────
        h = torch.cat([temporal_agg, long_bias, yr_emb, c_emb], dim=-1)
        return self.head(h).squeeze(-1)   # (B,)   y_norm prediction


# ============================================================================
# COUNTY STATS — auxiliary inputs only (not used to define target)
# ============================================================================
def compute_county_stats(samples):
    """
    Returns dict: FIPS → {'mean': float, 'std': float, 'trend': float}
    These are used ONLY as auxiliary input features to the model head.
    They do NOT define the training target.
    """
    from collections import defaultdict
    records = defaultdict(list)
    for s in samples:
        fips = str(s.get('FIPS', ''))
        year = int(s.get('year', 2010))
        y    = float(s['yield'])
        records[fips].append((year, y))

    stats = {}
    for fips, vals in records.items():
        vals.sort(key=lambda x: x[0])
        years  = np.array([v[0] for v in vals], dtype=np.float32)
        yields = np.array([v[1] for v in vals], dtype=np.float32)
        mean   = float(yields.mean())
        std    = float(yields.std()) if len(yields) > 1 else 1.0
        trend  = float(np.polyfit(years - years.mean(), yields, 1)[0]) \
                 if len(yields) > 1 else 0.0
        stats[fips] = {'mean': mean, 'std': max(std, 1.0), 'trend': trend}
    return stats


# ============================================================================
# DATASET
# ============================================================================
class YieldDataset(Dataset):
    def __init__(self, root_dir, json_file, config,
                 yield_mean=None, yield_std=None,
                 county_stats=None,
                 county_global_mean=None, county_global_std=None):
        self.root_dir           = root_dir
        self.config             = config
        self.yield_mean         = yield_mean   or 0.0
        self.yield_std          = yield_std    or 1.0
        self.county_stats       = county_stats or {}
        self.county_global_mean = county_global_mean or 48.0
        self.county_global_std  = county_global_std  or 15.0
        self.samples            = self._load_samples(json_file)

    def _load_samples(self, json_file):
        print(f'Loading {json_file}...')
        with open(json_file) as f:
            data = json.load(f)
        valid = []
        for s in tqdm(data, desc='Validating', leave=False):
            try:
                usda = os.path.join(self.root_dir, s['data']['USDA'])
                if not os.path.exists(usda): continue
                df = pd.read_csv(usda)
                if 'FIPS' not in df.columns:
                    df['FIPS'] = (df['state_ansi'].astype(str).str.zfill(2) +
                                  df['county_ansi'].astype(str).str.zfill(3))
                row = df[df['FIPS'] == str(s['FIPS'])]
                if len(row) == 0: continue
                y = float(row['YIELD, MEASURED IN BU / ACRE'].values[0])
                if pd.isna(y) or y <= 0: continue
                s['yield'] = y
                valid.append(s)
            except Exception:
                continue
        print(f'  ✅ {len(valid)} valid samples')
        return valid

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s        = self.samples[idx]
        sentinel = self._load_sentinel(s)
        hrrr_s   = self._load_hrrr_short(s)
        hrrr_l   = self._load_hrrr_long(s)
        y_raw    = float(s['yield'])

        # ── FIX: target is normalised raw yield, NOT anomaly ──────────
        y_norm = (y_raw - self.yield_mean) / self.yield_std

        # ── County features (aux inputs only) ─────────────────────────
        fips = str(s.get('FIPS', ''))
        cs   = self.county_stats.get(fips, {
            'mean': self.county_global_mean,
            'std':  self.county_global_std,
            'trend': 0.0})
        c_mean  = (cs['mean']  - self.county_global_mean) / self.county_global_std
        c_std   =  cs['std']   / self.county_global_std
        c_trend =  cs['trend'] / 5.0    # normalise trend ~0-5 bu/ac/yr
        county_feats = torch.tensor([c_mean, c_std, c_trend], dtype=torch.float32)

        # ── State ─────────────────────────────────────────────────────
        state_str = s.get('state', s.get('State', 'IA'))
        state_id  = STATE_MAP.get(state_str, 0)

        # ── Year ──────────────────────────────────────────────────────
        year_val = int(s.get('year', s.get('Year', 2010)))
        year_idx = max(0, min(29, year_val - Config.YEAR_BASE))

        # ── Sample weight ─────────────────────────────────────────────
        weight = Config.STATE_WEIGHTS.get(state_str, 1.0)

        return (sentinel, hrrr_s, hrrr_l,
                torch.tensor(y_norm,   dtype=torch.float32),   # train target
                torch.tensor(y_raw,    dtype=torch.float32),   # eval (denorm)
                torch.tensor(state_id, dtype=torch.long),
                torch.tensor(year_idx, dtype=torch.long),
                county_feats,
                torch.tensor(weight,   dtype=torch.float32),
                state_str)

    # ── Sentinel ──────────────────────────────────────────────────────────
    def _load_sentinel(self, sample):
        paths = sample['data'].get('Sentinel-2', {})
        if isinstance(paths, dict): paths = list(paths.values())
        if isinstance(paths, str):  paths = [paths]
        grids = []
        for p in paths[:self.config.NUM_GRIDS]:
            try:   grids.append(self._load_grid(self._fix(p)))
            except: grids.append(np.zeros((3, self.config.IMG_SIZE,
                                            self.config.IMG_SIZE), np.float32))
        while len(grids) < self.config.NUM_GRIDS:
            grids.append(np.zeros((3, self.config.IMG_SIZE,
                                   self.config.IMG_SIZE), np.float32))
        t = torch.from_numpy(
            np.stack(grids[:self.config.NUM_GRIDS])).float() / 10000.
        t = torch.clamp(t, 0, 1)
        return t.unsqueeze(0).repeat(self.config.NUM_SHORT_TERM, 1, 1, 1, 1)

    def _load_grid(self, path):
        if not os.path.exists(path):
            return np.zeros((3, self.config.IMG_SIZE, self.config.IMG_SIZE),
                            np.float32)
        with h5py.File(path, 'r') as f:
            k    = list(f.keys())
            data = f[k[0]][:]
        data = self._fix_shape(data)
        if data.shape[0] > 3: data = data[:3]
        elif data.shape[0] < 3:
            data = np.concatenate([data, np.zeros((3 - data.shape[0],
                                    data.shape[1], data.shape[2]),
                                    np.float32)])
        if (data.shape[1] != self.config.IMG_SIZE or
                data.shape[2] != self.config.IMG_SIZE):
            data = np.stack([cv2.resize(data[c].astype(np.float32),
                             (self.config.IMG_SIZE, self.config.IMG_SIZE))
                             for c in range(data.shape[0])])
        return data.astype(np.float32)

    @staticmethod
    def _fix_shape(d):
        while d.ndim > 3: d = d[0]
        if d.ndim == 2: d = d[np.newaxis]
        if d.ndim == 3 and d.shape[2] in (3, 4):
            d = np.transpose(d, (2, 0, 1))
        return d.astype(np.float32)

    # ── HRRR short ────────────────────────────────────────────────────────
    def _load_hrrr_short(self, sample):
        paths = (sample['data'].get('WRF-HRRR Computed Dataset', {})
                               .get('short_term', []))
        if isinstance(paths, str): paths = [paths]
        grids = []
        for p in paths[:self.config.NUM_GRIDS]:
            try:
                df    = pd.read_csv(self._fix(p))
                avail = [c for c in WEATHER_COLS if c in df.columns]
                v     = (df[avail].head(28).fillna(0).values
                         if avail else np.zeros((28, 9), np.float32))
                if v.shape[1] < 9:
                    v = np.pad(v, ((0, 0), (0, 9 - v.shape[1])))
                v = (v[:28] if len(v) >= 28
                     else np.pad(v, ((0, 28 - len(v)), (0, 0)))).astype(np.float32)
                grids.append(v)
            except Exception:
                grids.append(np.zeros((28, 9), np.float32))
        while len(grids) < self.config.NUM_GRIDS:
            grids.append(np.zeros((28, 9), np.float32))
        t = torch.from_numpy(np.stack(grids[:self.config.NUM_GRIDS])).float()
        t = (t - torch.tensor(W_MEAN).view(1, 1, -1)) / torch.tensor(W_STD).view(1, 1, -1)
        return t.unsqueeze(0).repeat(self.config.NUM_SHORT_TERM, 1, 1, 1)  # (T,G,28,9)

    # ── HRRR long ─────────────────────────────────────────────────────────
    def _load_hrrr_long(self, sample):
        long_data = (sample['data'].get('WRF-HRRR Computed Dataset', {})
                                   .get('long_term', []))
        result = []
        for year in long_data[:self.config.NUM_YEAR]:
            yr_vals = []
            if isinstance(year, list):
                for mp in year[:self.config.NUM_LONG_TERM]:
                    try:
                        df    = pd.read_csv(self._fix(mp))
                        avail = [c for c in WEATHER_COLS if c in df.columns]
                        v     = df[avail].mean(0).values if avail else np.zeros(9)
                        if len(v) < 9: v = np.pad(v, (0, 9 - len(v)))
                        yr_vals.append(v[:9].astype(np.float32))
                    except Exception:
                        yr_vals.append(np.zeros(9, np.float32))
            while len(yr_vals) < self.config.NUM_LONG_TERM:
                yr_vals.append(np.zeros(9, np.float32))
            result.append(yr_vals[:self.config.NUM_LONG_TERM])
        while len(result) < self.config.NUM_YEAR:
            result.append([np.zeros(9, np.float32)] * self.config.NUM_LONG_TERM)
        t = torch.from_numpy(np.array(result)).float()
        t = (t - torch.tensor(W_MEAN).view(1, 1, -1)) / torch.tensor(W_STD).view(1, 1, -1)
        return t

    def _fix(self, path):
        full = os.path.join(self.root_dir, path)
        if os.path.exists(full): return full
        parts = path.split('/')
        for i in range(len(parts)):
            p = os.path.join(self.root_dir, *parts[i:])
            if os.path.exists(p): return p
        return full


# ============================================================================
# UTILITIES
# ============================================================================
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def cosine_lr(optimizer, epoch, config):
    if epoch < config.WARMUP_EPOCHS:
        lr = config.LEARNING_RATE * (epoch + 1) / config.WARMUP_EPOCHS
    else:
        progress = ((epoch - config.WARMUP_EPOCHS) /
                    max(config.EPOCHS - config.WARMUP_EPOCHS, 1))
        lr = (config.MIN_LR + 0.5 * (config.LEARNING_RATE - config.MIN_LR)
              * (1 + math.cos(math.pi * progress)))
    for pg in optimizer.param_groups: pg['lr'] = lr
    return lr


def train_epoch(model, loader, optimizer, device):
    """
    FIX: trains on y_norm (normalised raw yield), not delta_norm (anomaly).
    """
    model.train()
    total = 0.0
    for batch in tqdm(loader, desc='Training', leave=False):
        (sentinel, hrrr_s, hrrr_l,
         y_norm, _,               # y_norm = (y_raw-MEAN)/STD  ← target
         state_ids, year_idx,
         county_feats, weights, _) = batch

        sentinel     = sentinel.to(device)
        hrrr_s       = hrrr_s.to(device)
        hrrr_l       = hrrr_l.to(device)
        y_norm       = y_norm.to(device)
        state_ids    = state_ids.to(device)
        year_idx     = year_idx.to(device)
        county_feats = county_feats.to(device)
        weights      = weights.to(device)

        optimizer.zero_grad()
        pred  = model(sentinel, hrrr_s, hrrr_l,
                      state_ids, year_idx, county_feats)
        # weighted Huber loss on normalised yield
        huber = F.huber_loss(pred, y_norm, reduction='none', delta=1.0)
        loss  = (huber * weights).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader, device, yield_mean, yield_std):
    """
    FIX: denormalise with YIELD_MEAN/YIELD_STD, not county mean/std.
    """
    model.eval()
    preds, trues, states = [], [], []
    for batch in tqdm(loader, desc='Evaluating', leave=False):
        (sentinel, hrrr_s, hrrr_l,
         _, y_raw,
         state_ids, year_idx,
         county_feats, _, state_str_list) = batch

        sentinel     = sentinel.to(device)
        hrrr_s       = hrrr_s.to(device)
        hrrr_l       = hrrr_l.to(device)
        state_ids    = state_ids.to(device)
        year_idx     = year_idx.to(device)
        county_feats = county_feats.to(device)

        pred_norm = model(sentinel, hrrr_s, hrrr_l,
                          state_ids, year_idx, county_feats).cpu()

        # FIX: simple denormalisation, no county reconstruction
        y_pred = pred_norm * yield_std + yield_mean
        preds.extend(y_pred.numpy())
        trues.extend(y_raw.numpy())
        states.extend(state_str_list)

    p  = np.array(preds)
    t  = np.array(trues)
    s  = np.array(states)
    rmse = float(np.sqrt(((p - t) ** 2).mean()))
    mae  = float(np.abs(p - t).mean())
    ss_r = float(np.sum((t - p) ** 2))
    ss_t = float(np.sum((t - t.mean()) ** 2))
    r2   = 1 - ss_r / ss_t if ss_t > 0 else 0.0
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        corr, _ = pearsonr(p, t)
    corr = float(corr) if not np.isnan(corr) else 0.0
    return rmse, mae, r2, corr, p, t, s


# ============================================================================
# PLOTS
# ============================================================================
SC = Config.STATE_COLORS

def _save(fig, path):
    fig.savefig(path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f'  📊 saved → {path}')


def _state_metrics(preds, targets, states_arr):
    metrics = []
    for st in Config.STATE_ORDER:
        mask = states_arr == st
        if mask.sum() == 0: continue
        p = preds[mask]; t = targets[mask]
        ss_r = float(np.sum((t - p) ** 2))
        ss_t = float(np.sum((t - t.mean()) ** 2))
        metrics.append({
            'state': st,
            'rmse':  float(np.sqrt(((p - t) ** 2).mean())),
            'mae':   float(np.abs(p - t).mean()),
            'r2':    1 - ss_r / ss_t if ss_t > 0 else 0.0,
            'n':     int(mask.sum()),
        })
    return metrics


def plot_training_curves(history, out_dir):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes[0, 0].plot(history['loss'], lw=2, color='#457b9d')
    axes[0, 0].set_title('Training Loss (Weighted Huber)')
    axes[0, 0].set_xlabel('Epoch'); axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(history['rmse'], lw=2, label='RMSE', color='#e63946')
    axes[0, 1].plot(history['mae'],  lw=2, label='MAE',  color='#f4a261')
    axes[0, 1].set_title('Validation Error (bu/acre)')
    axes[0, 1].set_xlabel('Epoch'); axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(history['r2'],   lw=2, label='R²',       color='#2a9d8f')
    axes[1, 0].plot(history['corr'], lw=2, label='Pearson r', color='#e9c46a')
    axes[1, 0].set_title('Validation Correlation')
    axes[1, 0].set_xlabel('Epoch'); axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(history['lr'], lw=2, color='#6a4c93')
    axes[1, 1].set_title('Learning Rate Schedule')
    axes[1, 1].set_xlabel('Epoch'); axes[1, 1].grid(alpha=0.3)

    plt.suptitle('CNN-Best V2.1 — Training Diagnostics', fontsize=14)
    plt.tight_layout()
    _save(fig, os.path.join(out_dir, 'plot_01_training_curves.png'))


def plot_pred_vs_actual(preds, targets, rmse, r2, corr, out_dir):
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(targets, preds, alpha=0.55, s=24, color='#457b9d')
    lims = [min(targets.min(), preds.min()) - 3,
            max(targets.max(), preds.max()) + 3]
    ax.plot(lims, lims, 'k--', lw=1.5)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Actual Yield (bu/acre)')
    ax.set_ylabel('Predicted Yield (bu/acre)')
    ax.set_title('Predicted vs Actual')
    ax.text(0.05, 0.95,
            f'RMSE = {rmse:.2f}\nR² = {r2:.4f}\nr = {corr:.4f}',
            transform=ax.transAxes, va='top', fontsize=11,
            bbox=dict(boxstyle='round', fc='white', alpha=0.8))
    ax.grid(alpha=0.3)
    plt.tight_layout()
    _save(fig, os.path.join(out_dir, 'plot_02_pred_vs_actual.png'))


def plot_residual_histogram(preds, targets, out_dir):
    err = preds - targets
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.hist(err, bins=30, color='#e63946', edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', lw=1.5, ls='--')
    ax.set_xlabel('Prediction Error (bu/acre)'); ax.set_ylabel('Count')
    ax.set_title(f'Residual Histogram  μ={err.mean():.2f}  σ={err.std():.2f}')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    _save(fig, os.path.join(out_dir, 'plot_03_residual_histogram.png'))


def plot_residuals_vs_actual(preds, targets, out_dir):
    err = preds - targets
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(targets, err, alpha=0.55, s=24, color='#2a9d8f')
    ax.axhline(0, color='black', ls='--', lw=1.5)
    ax.set_xlabel('Actual Yield (bu/acre)')
    ax.set_ylabel('Residual (Pred − Actual)')
    ax.set_title('Residuals vs Actual')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    _save(fig, os.path.join(out_dir, 'plot_04_residuals_vs_actual.png'))


def plot_per_state_rmse_mae(metrics, out_dir):
    states = [m['state'] for m in metrics]
    rmse   = [m['rmse']  for m in metrics]
    mae    = [m['mae']   for m in metrics]
    colors = [SC[s]      for s in states]
    x = np.arange(len(states)); w = 0.36
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.bar(x - w/2, rmse, w, label='RMSE', color=colors, alpha=0.9)
    ax.bar(x + w/2, mae,  w, label='MAE',  color=colors, alpha=0.55)
    ax.set_xticks(x); ax.set_xticklabels(states)
    ax.set_xlabel('State'); ax.set_ylabel('bu/acre')
    ax.set_title('Per-State RMSE and MAE')
    ax.legend(); ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    _save(fig, os.path.join(out_dir, 'plot_05_per_state_rmse_mae.png'))


def plot_per_state_r2(metrics, out_dir):
    states = [m['state'] for m in metrics]
    r2     = [m['r2']    for m in metrics]
    colors = [SC[s]      for s in states]
    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.bar(states, r2, color=colors)
    ax.axhline(0, color='black', lw=1.2, ls='--')
    ax.set_xlabel('State'); ax.set_ylabel('R²')
    ax.set_title('Per-State R²'); ax.grid(alpha=0.3, axis='y')
    for bar, val in zip(bars, r2):
        ypos = val + 0.02 if val >= 0 else val - 0.06
        ax.text(bar.get_x() + bar.get_width()/2, ypos,
                f'{val:.3f}', ha='center', fontsize=10)
    plt.tight_layout()
    _save(fig, os.path.join(out_dir, 'plot_06_per_state_r2.png'))


def plot_per_state_scatter(preds, targets, states_arr, out_dir):
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    for i, st in enumerate(Config.STATE_ORDER):
        ax   = axes[i]
        mask = states_arr == st
        if mask.sum() == 0: ax.axis('off'); continue
        p = preds[mask]; t = targets[mask]
        ax.scatter(t, p, alpha=0.6, s=22, color=SC[st])
        lims = [min(t.min(), p.min()) - 2, max(t.max(), p.max()) + 2]
        ax.plot(lims, lims, 'k--', lw=1.2)
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_title(f'{st}  (n={mask.sum()})')
        ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
        ax.grid(alpha=0.3)
    axes[-1].axis('off')
    plt.suptitle('Per-State Predicted vs Actual', fontsize=14)
    plt.tight_layout()
    _save(fig, os.path.join(out_dir, 'plot_07_per_state_scatter.png'))


def plot_error_boxplot_by_state(preds, targets, states_arr, out_dir):
    data, labels = [], []
    for st in Config.STATE_ORDER:
        mask = states_arr == st
        if mask.sum() == 0: continue
        data.append(preds[mask] - targets[mask]); labels.append(st)
    fig, ax = plt.subplots(figsize=(9, 6))
    bp = ax.boxplot(data, patch_artist=True, labels=labels)
    for patch, label in zip(bp['boxes'], labels):
        patch.set_facecolor(SC[label]); patch.set_alpha(0.6)
    ax.axhline(0, color='black', ls='--', lw=1.2)
    ax.set_xlabel('State'); ax.set_ylabel('Residual (Pred − Actual)')
    ax.set_title('Residual Distribution by State')
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    _save(fig, os.path.join(out_dir, 'plot_08_error_boxplot_by_state.png'))


def plot_sample_count(metrics, out_dir):
    states = [m['state'] for m in metrics]
    counts = [m['n']     for m in metrics]
    colors = [SC[s]      for s in states]
    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.bar(states, counts, color=colors)
    ax.set_xlabel('State'); ax.set_ylabel('Test Samples')
    ax.set_title('Sample Count by State'); ax.grid(alpha=0.3, axis='y')
    for bar, val in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.5,
                str(val), ha='center', va='bottom')
    plt.tight_layout()
    _save(fig, os.path.join(out_dir, 'plot_09_sample_count_by_state.png'))


def plot_season_centers(model, out_dir):
    mu_w  = model.season_mask.mu.detach().cpu().numpy()
    sig_w = model.season_mask.log_sigma.exp().detach().cpu().numpy()
    states = Config.STATE_ORDER

    x = np.arange(len(states))
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.bar(x, mu_w, yerr=sig_w, capsize=6,
           color=[SC[s] for s in states], alpha=0.8)
    ax.set_xticks(x); ax.set_xticklabels(states)
    ax.set_xlabel('State'); ax.set_ylabel('Season Center (week)')
    ax.set_title('Learned Season Centers ± Spread')
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    _save(fig, os.path.join(out_dir, 'plot_10_season_centers.png'))

    weeks = np.arange(1, 53)
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, st in enumerate(states):
        mu  = float(mu_w[i])
        sig = max(float(sig_w[i]), 0.1)
        curve = np.exp(-0.5 * ((weeks - mu) / sig) ** 2)
        curve /= curve.sum()
        ax.plot(weeks, curve, lw=2.5, label=st, color=SC[st])
    ax.set_xlabel('Week of Year'); ax.set_ylabel('Normalized Weight')
    ax.set_title('Season Mask Attention Curves')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    _save(fig, os.path.join(out_dir, 'plot_11_season_mask_curves.png'))


def plot_summary_dashboard(history, preds, targets, states_arr, metrics,
                           rmse, r2, corr, out_dir):
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))

    axes[0, 0].plot(history['rmse'], lw=2, label='RMSE', color='#e63946')
    axes[0, 0].plot(history['mae'],  lw=2, label='MAE',  color='#f4a261')
    axes[0, 0].set_title('Validation Error Across Epochs')
    axes[0, 0].set_xlabel('Epoch'); axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)

    lims = [min(targets.min(), preds.min()) - 3,
            max(targets.max(), preds.max()) + 3]
    axes[0, 1].scatter(targets, preds, alpha=0.45, s=20, color='#457b9d')
    axes[0, 1].plot(lims, lims, 'k--', lw=1.2)
    axes[0, 1].set_xlim(lims); axes[0, 1].set_ylim(lims)
    axes[0, 1].set_title(f'Pred vs Actual  RMSE={rmse:.2f}  R²={r2:.3f}  r={corr:.3f}')
    axes[0, 1].set_xlabel('Actual'); axes[0, 1].set_ylabel('Predicted')
    axes[0, 1].grid(alpha=0.3)

    st_list = [m['state'] for m in metrics]
    r2_vals = [m['r2']    for m in metrics]
    axes[1, 0].bar(st_list, r2_vals, color=[SC[s] for s in st_list])
    axes[1, 0].axhline(0, color='black', ls='--', lw=1.)
    axes[1, 0].set_title('Per-State R²')
    axes[1, 0].set_xlabel('State'); axes[1, 0].set_ylabel('R²')
    axes[1, 0].grid(alpha=0.3, axis='y')

    data, labels = [], []
    for st in Config.STATE_ORDER:
        mask = states_arr == st
        if mask.sum() == 0: continue
        data.append(preds[mask] - targets[mask]); labels.append(st)
    bp = axes[1, 1].boxplot(data, patch_artist=True, labels=labels)
    for patch, label in zip(bp['boxes'], labels):
        patch.set_facecolor(SC[label]); patch.set_alpha(0.6)
    axes[1, 1].axhline(0, color='black', ls='--', lw=1.)
    axes[1, 1].set_title('Residuals by State')
    axes[1, 1].set_xlabel('State'); axes[1, 1].set_ylabel('Pred − Actual')
    axes[1, 1].grid(alpha=0.3, axis='y')

    plt.suptitle('CNN-Best V2.1 — Performance Dashboard', fontsize=15)
    plt.tight_layout()
    _save(fig, os.path.join(out_dir, 'plot_12_summary_dashboard.png'))


# ============================================================================
# MAIN
# ============================================================================
def main():
    set_seed(Config.SEED)
    os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
    CKPT = os.path.join(Config.OUTPUT_DIR, 'best_model.pth')

    print('=' * 70)
    print('  CNN-Best V2.1  —  Fixed: raw-yield target + forward fusion')
    print('=' * 70)
    print(f'  Device  : {Config.DEVICE}')
    print(f'  Epochs  : {Config.EPOCHS}  (patience={Config.PATIENCE})')
    print(f'  Batch   : {Config.BATCH_SIZE}')
    print('=' * 70)

    # ── [1] train stats ───────────────────────────────────────────────
    print('\n[1/5] Computing stats from train set...')
    tmp_ds = YieldDataset(Config.ROOT_DIR, Config.TRAIN_JSON, Config)
    raw    = np.array([s['yield'] for s in tmp_ds.samples])
    YIELD_MEAN = float(raw.mean())
    YIELD_STD  = float(raw.std())
    print(f'  yield mean={YIELD_MEAN:.2f}  std={YIELD_STD:.2f}  '
          f'counties={len(tmp_ds.samples)}')

    county_stats = compute_county_stats(tmp_ds.samples)
    c_means = np.array([v['mean'] for v in county_stats.values()])
    COUNTY_GLOBAL_MEAN = float(c_means.mean())
    COUNTY_GLOBAL_STD  = float(c_means.std() + 1e-6)

    # ── [2] datasets ──────────────────────────────────────────────────
    print('\n[2/5] Building datasets...')
    kw = dict(config=Config,
              yield_mean=YIELD_MEAN, yield_std=YIELD_STD,
              county_stats=county_stats,
              county_global_mean=COUNTY_GLOBAL_MEAN,
              county_global_std=COUNTY_GLOBAL_STD)
    train_ds = YieldDataset(Config.ROOT_DIR, Config.TRAIN_JSON, **kw)
    test_ds  = YieldDataset(Config.ROOT_DIR, Config.TEST_JSON,  **kw)
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE,
                              shuffle=True,  num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=Config.BATCH_SIZE,
                              shuffle=False, num_workers=4, pin_memory=True)

    # ── [3] model ─────────────────────────────────────────────────────
    print('\n[3/5] Building model...')
    model = CNNBestV2(Config).to(Config.DEVICE)
    n_par = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Parameters: {n_par/1e6:.2f}M')

    optimizer = optim.AdamW(model.parameters(),
                            lr=Config.LEARNING_RATE,
                            betas=(0.9, 0.95),
                            weight_decay=Config.WEIGHT_DECAY)

    # ── [4] training ──────────────────────────────────────────────────
    print('\n[4/5] Training...\n')
    best_rmse    = float('inf')
    patience_cnt = 0
    history = {'loss': [], 'rmse': [], 'mae': [], 'r2': [], 'corr': [], 'lr': []}

    for epoch in range(Config.EPOCHS):
        lr         = cosine_lr(optimizer, epoch, Config)
        train_loss = train_epoch(model, train_loader, optimizer, Config.DEVICE)
        rmse, mae, r2, corr, preds, targets, states = evaluate(
            model, test_loader, Config.DEVICE, YIELD_MEAN, YIELD_STD)

        history['loss'].append(train_loss)
        history['rmse'].append(rmse); history['mae'].append(mae)
        history['r2'].append(r2);     history['corr'].append(corr)
        history['lr'].append(lr)

        flag = ''
        if rmse < best_rmse:
            best_rmse    = rmse
            patience_cnt = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'rmse': rmse, 'r2': r2,
                'yield_mean': YIELD_MEAN, 'yield_std': YIELD_STD,
            }, CKPT)
            flag = ' ✅ best'
        else:
            patience_cnt += 1

        print(f'Ep {epoch:>3} | Loss {train_loss:.4f} | '
              f'RMSE {rmse:.2f} | MAE {mae:.2f} | '
              f'R² {r2:.4f} | r {corr:.4f} | '
              f'LR {lr:.2e}{flag}')

        if patience_cnt >= Config.PATIENCE:
            print(f'\n⏹  Early stopping at epoch {epoch}')
            break

    # ── [5] final evaluation ──────────────────────────────────────────
    print('\n[5/5] Final evaluation...')
    ckpt = torch.load(CKPT, map_location=Config.DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    rmse, mae, r2, corr, preds, targets, states = evaluate(
        model, test_loader, Config.DEVICE, YIELD_MEAN, YIELD_STD)
    metrics = _state_metrics(preds, targets, states)

    print(f'\n{"="*70}')
    print('   FINAL RESULTS — CNN-Best V2.1')
    print(f'{"="*70}')
    print(f'  RMSE      : {rmse:.4f} bu/acre')
    print(f'  MAE       : {mae:.4f} bu/acre')
    print(f'  R²        : {r2:.4f}')
    print(f'  Pearson r : {corr:.4f}')
    print()
    print(f'   {"State":>5} | {"RMSE":>6} | {"MAE":>6} | {"R²":>8} | {"n":>5}')
    print('  ' + '-' * 42)
    for m in metrics:
        print(f'  {m["state"]:>5} | {m["rmse"]:>6.2f} | {m["mae"]:>6.2f} | '
              f'{m["r2"]:>8.4f} | {m["n"]:>5}')

    mu_w  = model.season_mask.mu.detach().cpu().numpy()
    sig_w = model.season_mask.log_sigma.exp().detach().cpu().numpy()
    MONTH = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
    print()
    print('  Learned season centers (SeasonMaskV2):')
    print(f'   {"State":>5} | {"μ (week)":>9} | {"σ (weeks)":>10} | {"Peak month":>11}')
    print('  ' + '-' * 46)
    for i, st in enumerate(Config.STATE_ORDER):
        peak_month = MONTH.get(int(mu_w[i] // 4.33) + 1, '?')
        print(f'  {st:>5} | {mu_w[i]:>9.1f} | {sig_w[i]:>10.1f} | {peak_month:>11}')

    print(f'\n  Parameters  : {n_par/1e6:.2f}M  (MMST-ViT was 43.2M)')
    print(f'  Best epoch  : {ckpt["epoch"]}')
    print(f'  Outputs     : {Config.OUTPUT_DIR}')
    print(f'{"="*70}')

    # ── plots ─────────────────────────────────────────────────────────
    print('\nGenerating all plots...')
    plot_training_curves(history, Config.OUTPUT_DIR)
    plot_pred_vs_actual(preds, targets, rmse, r2, corr, Config.OUTPUT_DIR)
    plot_residual_histogram(preds, targets, Config.OUTPUT_DIR)
    plot_residuals_vs_actual(preds, targets, Config.OUTPUT_DIR)
    plot_per_state_rmse_mae(metrics, Config.OUTPUT_DIR)
    plot_per_state_r2(metrics, Config.OUTPUT_DIR)
    plot_per_state_scatter(preds, targets, states, Config.OUTPUT_DIR)
    plot_error_boxplot_by_state(preds, targets, states, Config.OUTPUT_DIR)
    plot_sample_count(metrics, Config.OUTPUT_DIR)
    plot_season_centers(model, Config.OUTPUT_DIR)
    plot_summary_dashboard(history, preds, targets, states, metrics,
                           rmse, r2, corr, Config.OUTPUT_DIR)
    print(f'\n✅ All plots saved to {Config.OUTPUT_DIR}')


if __name__ == '__main__':
    main()

  CNN-Best V2.1  —  Fixed: raw-yield target + forward fusion
  Device  : cuda
  Epochs  : 50  (patience=20)
  Batch   : 12

[1/5] Computing stats from train set...
Loading /workspace/MMST-ViT/data/soybean_train.json...


  ✅ 344 valid samples
  yield mean=48.52  std=15.85  counties=344

[2/5] Building datasets...
Loading /workspace/MMST-ViT/data/soybean_train.json...


  ✅ 344 valid samples
Loading /workspace/MMST-ViT/data/soybean_test.json...


  ✅ 327 valid samples

[3/5] Building model...
  Parameters: 4.14M

[4/5] Training...



Ep   0 | Loss 0.5074 | RMSE 10.17 | MAE 8.58 | R² 0.3889 | r 0.8927 | LR 3.00e-05 ✅ best


...

Ep  25 | Loss 0.0059 | RMSE 6.20 | MAE 4.78 | R² 0.7727 | r 0.8977 | LR 2.08e-04


Ep  26 | Loss 0.0068 | RMSE 6.80 | MAE 5.13 | R² 0.7269 | r 0.8983 | LR 1.97e-04


Ep  27 | Loss 0.0058 | RMSE 6.83 | MAE 5.17 | R² 0.7246 | r 0.8956 | LR 1.85e-04


Ep  28 | Loss 0.0055 | RMSE 6.42 | MAE 4.96 | R² 0.7568 | r 0.8929 | LR 1.74e-04

⏹  Early stopping at epoch 28

[5/5] Final evaluation...



   FINAL RESULTS — CNN-Best V2.1
  RMSE      : 5.7606 bu/acre
  MAE       : 4.6351 bu/acre
  R²        : 0.8040
  Pearson r : 0.8970

   State |   RMSE |    MAE |       R² |     n
  ------------------------------------------
     IA |   6.26 |   4.78 |   0.2712 |    75
     IL |   4.98 |   4.23 |   0.5414 |    74
     MN |   6.06 |   4.70 |   0.5636 |    66
     NC |   5.68 |   4.83 |   0.1515 |    71
     ND |   5.76 |   4.66 |   0.3799 |    41

  Learned season centers (SeasonMaskV2):
   State |  μ (week) |  σ (weeks) |  Peak month
  ----------------------------------------------
     IA |      25.9 |        1.0 |         Jun
     IL |      25.9 |        1.0 |         Jun
     MN |      23.9 |        1.0 |         Jun
     NC |      29.9 |        1.0 |         Jul
     ND |      22.9 |        1.0 |         Jun

  Parameters  : 4.14M  (MMST-ViT was 43.2M)
  Best epoch  : 8
  Outputs     : /workspace/MMST-ViT/output_dir/cnn_best_v2_1

Generating all plots...
  📊 saved → /workspace/MMS

In [50]:
set_seed(Config.SEED)

ckpt = torch.load(CKPT_PATH, map_location=Config.DEVICE, weights_only=False)
YIELD_MEAN = float(ckpt['yield_mean'])
YIELD_STD  = float(ckpt['yield_std'])

model = CNNBestV2(Config).to(Config.DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print(f'Best epoch : {ckpt["epoch"]}')
print(f'Yield mean : {YIELD_MEAN:.2f}  std: {YIELD_STD:.2f}')
print()
print(model)

Best epoch : 8
Yield mean : 48.52  std: 15.85

CNNBestV2(
  (image_encoder): ImageEncoder(
    (stem): Sequential(
      (0): Conv2d(3, 32, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    )
    (layer1): ConvBlock(
      (conv): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (skip): Sequential(
        (0): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d

In [ ]:
# Show top-level attribute names 
for name, module in model.named_children():
    print(name, '->', type(module).__name__)

image_encoder -> ImageEncoder
weather_1d_cnn -> Weather1DCNN
weather_fusion -> WeatherFusion
spatial_pool -> SpatialPool
long_term_enc -> Sequential
temporal_gru -> TemporalGRU
season_mask -> SeasonMaskV2
year_embed -> Embedding
county_proj -> Sequential
head -> PredictionHead


In [53]:
for name, module in model.image_encoder.named_children():
    print(name, '->', type(module).__name__)

stem -> Sequential
layer1 -> ConvBlock
layer2 -> ConvBlock
layer3 -> ConvBlock
layer4 -> ConvBlock
pool -> AdaptiveAvgPool2d
norm -> LayerNorm
